# AlignPre and AlignPost projections in ``brainstate``

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaobrain/brainx/blob/main/docs/tutorials/brainstate_alignpre_aignpost.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/chaobrain/brainx/blob/main/docs/tutorials/brainstate_alignpre_aignpost.ipynb)

This tutorial explains the idea behind **state alignment** in event-driven projections. A projection has four parts:

- a presynaptic event stream, usually spikes;
- a communication operator, such as sparse random connectivity;
- a synaptic state, such as an exponential conductance or current trace;
- a postsynaptic target that receives the projected output.

The central question is where the synaptic state should live. **AlignPre** keeps the state on the presynaptic side before communication. **AlignPost** keeps the state on the postsynaptic side after communication. Both describe the same modeling ingredients, but they optimize different network structures.

You will learn how to:

- reason about pre-aligned and post-aligned state shapes;
- build a small numerical example that shows the data flow;
- implement a BrainState/BrainPy excitatory-inhibitory network with `brainpy.state.AlignPostProj`;
- choose the alignment pattern that matches a model architecture.

Prerequisites:

- basic Python and NumPy;
- familiarity with spikes, synaptic traces, and leaky integrate-and-fire neurons;
- `brainstate`, `brainunit`, `brainpy.state`, and `braintools` installed for the package-level example.

## Alignment in one picture

Suppose a presynaptic population of size `n_pre` projects to a postsynaptic population of size `n_post` through a weight matrix `W`.

In a **pre-aligned** projection, the synaptic trace has shape `(n_pre,)`:

```text
pre_spike -> pre_trace -> communicate with W -> post_input
```

This is useful when one presynaptic source fans out to multiple targets, because the same presynaptic trace can be reused.

In a **post-aligned** projection, the communication happens first and the synaptic trace has shape `(n_post,)`:

```text
pre_spike -> communicate with W -> post_trace -> post_input
```

This is useful when multiple sources converge on one target population, because all incoming events can be accumulated into the target-side synaptic state.

## Setup for the alignment toy model

The first part uses only NumPy and Matplotlib. It is not meant to replace the package projection classes; it makes the tensor shapes and update order visible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(12)

n_steps = 80
n_pre = 8
n_post_a = 5
n_post_b = 4

dt_ms = 0.1
tau_ms = 5.0
decay = np.exp(-dt_ms / tau_ms)

# A sparse Bernoulli spike train from the presynaptic population.
pre_spikes = rng.random((n_steps, n_pre)) < 0.05

# Two independent targets receive input from the same presynaptic source.
w_a = (rng.random((n_pre, n_post_a)) < 0.45) * rng.uniform(0.2, 0.8, (n_pre, n_post_a))
w_b = (rng.random((n_pre, n_post_b)) < 0.45) * rng.uniform(0.2, 0.8, (n_pre, n_post_b))

print("pre_spikes:", pre_spikes.shape)
print("w_a:", w_a.shape)
print("w_b:", w_b.shape)

## Pre-aligned trace: update once, reuse many times

A pre-aligned trace is attached to the presynaptic population. Each time step updates one trace per presynaptic neuron:

```text
pre_trace[t + 1] = decay * pre_trace[t] + pre_spike[t]
```

After that, the same trace can be projected into any number of downstream targets.

In [ ]:
pre_trace = np.zeros(n_pre)
pre_trace_history = np.zeros((n_steps, n_pre))

for i, spike in enumerate(pre_spikes.astype(float)):
    pre_trace = decay * pre_trace + spike
    pre_trace_history[i] = pre_trace

# Reuse the same presynaptic trace for two targets.
drive_a_from_pre = pre_trace_history @ w_a
drive_b_from_pre = pre_trace_history @ w_b

print("pre_trace_history:", pre_trace_history.shape)
print("drive_a_from_pre:", drive_a_from_pre.shape)
print("drive_b_from_pre:", drive_b_from_pre.shape)

In [ ]:
time_ms = np.arange(n_steps) * dt_ms

fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)

axes[0].imshow(pre_spikes.T, aspect="auto", interpolation="nearest", cmap="Greys")
axes[0].set_ylabel("pre neuron")
axes[0].set_title("Presynaptic spikes")

axes[1].plot(time_ms, pre_trace_history[:, :3])
axes[1].set_ylabel("trace")
axes[1].set_title("Pre-aligned traces for three presynaptic neurons")

axes[2].plot(time_ms, drive_a_from_pre[:, 0], label="target A neuron 0")
axes[2].plot(time_ms, drive_b_from_pre[:, 0], label="target B neuron 0")
axes[2].set_xlabel("time (ms)")
axes[2].set_ylabel("drive")
axes[2].set_title("The same pre trace fans out to two targets")
axes[2].legend(loc="best")

plt.tight_layout()
plt.show()

## Post-aligned trace: communicate first, integrate on the target

A post-aligned trace first maps presynaptic events into the target population. The trace then has one state value per postsynaptic neuron:

```text
incoming[t] = pre_spike[t] @ W
post_trace[t + 1] = decay * post_trace[t] + incoming[t]
```

This is the natural layout when the target population owns the synaptic current or conductance state.

In [ ]:
incoming_a = pre_spikes.astype(float) @ w_a
post_trace_a = np.zeros(n_post_a)
post_trace_history_a = np.zeros((n_steps, n_post_a))

for i, incoming in enumerate(incoming_a):
    post_trace_a = decay * post_trace_a + incoming
    post_trace_history_a[i] = post_trace_a

print("incoming_a:", incoming_a.shape)
print("post_trace_history_a:", post_trace_history_a.shape)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)

axes[0].plot(time_ms, incoming_a[:, :3])
axes[0].set_ylabel("event input")
axes[0].set_title("Events after communication into target A")

axes[1].plot(time_ms, post_trace_history_a[:, :3])
axes[1].set_xlabel("time (ms)")
axes[1].set_ylabel("trace")
axes[1].set_title("Post-aligned traces for three postsynaptic neurons")

plt.tight_layout()
plt.show()

## What changes between the two layouts?

The toy model used the same spikes and weights, but the state placement changed the computation order.

| Question | AlignPre | AlignPost |
| --- | --- | --- |
| State shape | `n_pre` | `n_post` |
| First operation | update presynaptic trace | communicate presynaptic events |
| Best fit | one source fans out to several targets | many sources converge on one target |
| Main reuse | reuse a pre trace across projections | reuse target-side synaptic/output state |
| Typical failure mode | duplicating traces when one pre source feeds many targets | duplicating target-side state when many inputs share the same target |

The right choice is not about biological correctness by itself. It is about where the mathematical state belongs and which layout avoids redundant work for the network you are building.

## A BrainState/BrainPy `AlignPostProj` network

The package-level example below follows the same style as the EI network tutorials in this documentation. We build one leaky integrate-and-fire population and two post-aligned projections:

- an excitatory projection from the excitatory subpopulation to all neurons;
- an inhibitory projection from the inhibitory subpopulation to all neurons.

Both projections use `brainpy.state.AlignPostProj`, so each synaptic trace is aligned to the shared postsynaptic population.

In [ ]:
import brainunit as u
import brainstate
import braintools
import brainpy.state

brainstate.environ.set(dt=0.1 * u.ms)

In [ ]:
class EINet(brainstate.nn.Module):
    """A compact excitatory-inhibitory network using post-aligned projections."""

    def __init__(self, n_exc, n_inh, prob, exc_weight, inh_weight):
        super().__init__()
        self.n_exc = n_exc
        self.n_inh = n_inh
        self.num = n_exc + n_inh

        self.neurons = brainpy.state.LIF(
            self.num,
            V_rest=-52.0 * u.mV,
            V_th=-50.0 * u.mV,
            V_reset=-60.0 * u.mV,
            tau=10.0 * u.ms,
            V_initializer=braintools.init.Normal(-60.0, 5.0, unit=u.mV),
            spk_reset="soft",
        )

        self.exc = brainpy.state.AlignPostProj(
            comm=brainstate.nn.EventFixedProb(n_exc, self.num, prob, exc_weight),
            syn=brainpy.state.Expon.desc(self.num, tau=2.0 * u.ms),
            out=brainpy.state.CUBA.desc(),
            post=self.neurons,
        )
        self.inh = brainpy.state.AlignPostProj(
            comm=brainstate.nn.EventFixedProb(n_inh, self.num, prob, inh_weight),
            syn=brainpy.state.Expon.desc(self.num, tau=5.0 * u.ms),
            out=brainpy.state.CUBA.desc(),
            post=self.neurons,
        )

    def update(self, background_current):
        spikes = self.neurons.get_spike() != 0.0
        self.exc(spikes[:self.n_exc])
        self.inh(spikes[self.n_exc:])
        self.neurons(background_current)
        return self.neurons.get_spike()

### Run the network

The excitatory and inhibitory weights are scaled by population size so the example stays stable when you change `n_exc`, `n_inh`, or `prob`. The exact activity pattern is not the point of the tutorial; the important part is the projection layout in `EINet.__init__`.

In [ ]:
n_exc = 320
n_inh = 80
prob = 0.05

exc_weight = 1.0 / u.math.sqrt(prob * n_exc) * u.mS
inh_weight = -1.2 / u.math.sqrt(prob * n_inh) * u.mS
background_current = 3.0 * u.mA

net = EINet(n_exc, n_inh, prob, exc_weight, inh_weight)
_ = brainstate.nn.init_all_states(net)

duration = 300.0 * u.ms
times = u.math.arange(0.0 * u.ms, duration, brainstate.environ.get_dt())

spikes = brainstate.transform.for_loop(
    lambda t: net.update(background_current),
    times,
    pbar=brainstate.transform.ProgressBar(20),
)

print("spikes:", spikes.shape)

In [ ]:
t_indices, neuron_indices = u.math.where(spikes)

plt.figure(figsize=(8, 4))
plt.scatter(times[t_indices], neuron_indices, s=1, c="black")
plt.axhline(n_exc - 0.5, color="tab:red", lw=0.8, label="E/I boundary")
plt.xlabel("time")
plt.ylabel("neuron index")
plt.title("Spike raster from the post-aligned EI network")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Reading the `AlignPostProj` code

The projection constructor separates the modeling pieces clearly:

- `comm=brainstate.nn.EventFixedProb(...)` creates an event-driven sparse communication operator from a presynaptic slice to the full postsynaptic population.
- `syn=brainpy.state.Expon.desc(self.num, tau=...)` declares an exponential synaptic state with shape `self.num`, so the state is aligned to the postsynaptic population.
- `out=brainpy.state.CUBA.desc()` converts the synaptic state into a current-based output.
- `post=self.neurons` attaches the output to the target population.

The update order is also deliberate. The projection consumes the previous-step spikes, writes synaptic input into the target, and then the neuron population advances one step with the background current.

## Choosing an alignment pattern

Use **AlignPre** when the expensive or meaningful state belongs to the presynaptic side. Common examples include a presynaptic trace reused by several outgoing projections, short-term plasticity variables attached to presynaptic terminals, or a fan-out architecture where recomputing the same trace for each target would be wasteful.

Use **AlignPost** when the state belongs to the target side. Common examples include conductance/current traces accumulated by a postsynaptic population, multiple inputs converging into one target, or output models that need access to postsynaptic variables such as voltage.

A practical rule is to write down the state shape before choosing the class or helper:

```text
state shape == n_pre   -> pre-aligned design
state shape == n_post  -> post-aligned design
```

If the shape is unclear, sketch the update equations first. The correct projection layout usually becomes obvious once every state variable has an owner.

## Exercises

1. Change `tau` in the excitatory and inhibitory `Expon.desc(...)` calls. How does the raster change when inhibition is slower than excitation?
2. Increase `prob` while reducing the weights. Does the network become smoother or more synchronous?
3. Split the excitatory population into two presynaptic groups that both project to `self.neurons`. This is still a post-aligned design because both inputs accumulate into the same target-side state.
4. Return to the NumPy fan-out example and add a third target. The pre-aligned trace should still be computed only once.

## Troubleshooting

- **Shape mismatch:** check whether the synaptic state is sized by `n_pre` or `n_post`. In the `AlignPostProj` example, `Expon.desc(self.num, ...)` must match the postsynaptic population.
- **No spikes:** increase `background_current`, increase `exc_weight`, or run for a longer duration.
- **Runaway activity:** reduce excitation, strengthen inhibition, lower connection probability, or shorten the excitatory synaptic time constant.
- **Unit errors:** keep times in units such as `u.ms`, voltages in `u.mV`, currents in `u.mA`, and conductances in `u.mS`. Let `brainunit` catch incompatible equations early.